### E-Commerce Recommendation System

#### Milestone 1

In [1]:
import pandas as pd

# Define chunk size to avoid memory issues
chunk_size = 10_000

interaction_chunks = []

# Map event types to numeric interaction scores
event_strength = {'view': 1, 'addtocart': 2, 'transaction': 3}

chunks = pd.read_csv('/content/events.csv', chunksize=chunk_size)

for i, chunk in enumerate(chunks):
    # Drop rows with missing visitorid or itemid
    chunk = chunk.dropna(subset=['visitorid', 'itemid'])

    # Map event type to interaction strength
    chunk['interaction'] = chunk['event'].map(event_strength).fillna(0).astype(int)

    # Filter out interactions with zero strength (if any)
    chunk = chunk[chunk['interaction'] > 0]

    # Keep only relevant columns
    chunk = chunk[['visitorid', 'itemid', 'interaction']]

    interaction_chunks.append(chunk)

    if i == 4:  # Use only first 5 chunks to limit memory usage
        break

# Concatenate chunks into a single DataFrame
interactions_df = pd.concat(interaction_chunks)

# Build user-item interaction matrix (pivot table)
user_item_matrix = interactions_df.pivot_table(
    index='visitorid',
    columns='itemid',
    values='interaction',
    aggfunc='sum',
    fill_value=0
)

print(user_item_matrix.shape)
print(user_item_matrix.head())


(28860, 24716)
itemid     6       15      55      66      92      147     168     190     \
visitorid                                                                   
17              0       0       0       0       0       0       0       0   
52              0       0       0       0       0       0       0       0   
74              0       0       0       0       0       0       0       0   
109             0       0       0       0       0       0       0       0   
122             0       0       0       0       0       0       0       0   

itemid     195     217     ...  466760  466772  466785  466789  466795  \
visitorid                  ...                                           
17              0       0  ...       0       0       0       0       0   
52              0       0  ...       0       0       0       0       0   
74              0       0  ...       0       0       0       0       0   
109             0       0  ...       0       0       0       0       0   
1

#### Milestone 2

In [2]:
# Remove very inactive users
user_item_matrix = user_item_matrix[user_item_matrix.sum(axis=1) >= 2]

# Remove very unpopular items
user_item_matrix = user_item_matrix.loc[:, user_item_matrix.sum(axis=0) >= 2]

print("Filtered matrix shape:", user_item_matrix.shape)

Filtered matrix shape: (7297, 6036)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Compute item-item similarity
item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

print("Item similarity matrix shape:", item_similarity_df.shape)


Item similarity matrix shape: (6036, 6036)


In [6]:
def recommend_items(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found"

    user_vector = user_item_matrix.loc[user_id]

    # Score items based on similarity
    scores = item_similarity_df.dot(user_vector)

    # Remove items already interacted with
    scores[user_vector > 0] = 0

    # Sort and return top-N
    return scores.sort_values(ascending=False).head(top_n)


In [7]:
sample_user = user_item_matrix.index[0]
recommend_items(sample_user, top_n=5)


,0
itemid,
140461,0.230089
310764,0.000000
310754,0.000000
310725,0.000000
310720,0.000000


##### Tuning

In [8]:
# Remove users with zero total interaction
user_item_matrix = user_item_matrix[user_item_matrix.sum(axis=1) > 0]

# Remove items with zero total interaction
user_item_matrix = user_item_matrix.loc[:, user_item_matrix.sum(axis=0) > 0]

print("After zero-removal shape:", user_item_matrix.shape)


After zero-removal shape: (6383, 6036)


In [9]:
# -------- TUNING: NORMALIZATION --------

# Compute row sums
row_sums = user_item_matrix.sum(axis=1)

# Keep only users with non-zero interactions
user_item_matrix_nonzero = user_item_matrix[row_sums > 0]

# Normalize safely
user_item_matrix_norm = user_item_matrix_nonzero.div(
    user_item_matrix_nonzero.sum(axis=1), axis=0
)

print("NaN check:", user_item_matrix_norm.isna().sum().sum())
print("Normalized matrix shape:", user_item_matrix_norm.shape)


NaN check: 0
Normalized matrix shape: (6383, 6036)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

item_similarity_norm = cosine_similarity(user_item_matrix_norm.T)

item_similarity_norm_df = pd.DataFrame(
    item_similarity_norm,
    index=user_item_matrix_norm.columns,
    columns=user_item_matrix_norm.columns
)

print("Tuned (normalized) model trained successfully.")


Tuned (normalized) model trained successfully.


In [11]:
def recommend_items_norm(user_id, top_n=5):
    if user_id not in user_item_matrix_norm.index:
        return "User not found"

    user_vector = user_item_matrix_norm.loc[user_id]
    scores = item_similarity_norm_df.dot(user_vector)

    scores[user_vector > 0] = 0
    return scores.sort_values(ascending=False).head(top_n)


In [12]:
print("Baseline recommendations:")
print(recommend_items(sample_user))

print("\nTuned recommendations:")
print(recommend_items_norm(sample_user))


Baseline recommendations:
itemid
140461    0.230089
310764    0.000000
310754    0.000000
310725    0.000000
310720    0.000000
dtype: float64

Tuned recommendations:
itemid
140461    0.447128
310764    0.000000
310754    0.000000
310725    0.000000
310720    0.000000
dtype: float64


#### **Milestone 3**

In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def precision_at_k(recommended_items, actual_items, k):

    # Handle edge case
    if k == 0:
        return 0.0

    # Get only the top K recommendations
    top_k_recommendations = recommended_items[:k]

    # Count how many recommended items match actual items
    hits = 0
    for item in top_k_recommendations:
        if item in actual_items:
            hits += 1

    # Calculate precision
    precision = hits / k
    return precision

In [15]:
def recall_at_k(recommended_items, actual_items, k):

    # Handle edge case
    if len(actual_items) == 0:
        return 0.0

    # Get only the top K recommendations
    top_k_recommendations = recommended_items[:k]

    # Count how many recommended items match actual items
    hits = 0
    for item in top_k_recommendations:
        if item in actual_items:
            hits += 1

    # Calculate recall
    recall = hits / len(actual_items)
    return recall

In [17]:
def f1_score_at_k(precision, recall):

    # Handle edge case
    if precision + recall == 0:
        return 0.0

    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


print("Evaluation functions defined successfully!")


Evaluation functions defined successfully!


In [19]:
# Split data into training and test set

def split_data_for_evaluation(user_item_matrix, test_ratio=0.2):

    # Create copies for train and test
    train_matrix = user_item_matrix.copy()
    test_matrix = user_item_matrix.copy()

    # Dictionary to store which items go where for each user
    user_splits = {}

    for user_id in user_item_matrix.index:
        # Find all items this user interacted with
        user_row = user_item_matrix.loc[user_id]
        items_user_liked = user_row[user_row > 0].index.tolist()

        # If user has no interactions, skip
        if len(items_user_liked) == 0:
            user_splits[user_id] = {'train': [], 'test': []}
            continue

        # Calculate how many items go to test set (at least 1)
        num_test_items = max(1, int(len(items_user_liked) * test_ratio))

        # Randomly pick items for test set
        test_items = np.random.choice(items_user_liked, size=num_test_items, replace=False)
        test_items = test_items.tolist()

        # Remaining items go to training set
        train_items = [item for item in items_user_liked if item not in test_items]

        # Save the split
        user_splits[user_id] = {
            'train': train_items,
            'test': test_items
        }

        # Remove test items from training matrix
        train_matrix.loc[user_id, test_items] = 0

        # Remove train items from test matrix
        test_matrix.loc[user_id, train_items] = 0

    print(f"  Train matrix shape: {train_matrix.shape}")
    print(f"  Test matrix shape: {test_matrix.shape}")

    return train_matrix, test_matrix, user_splits


# Split the data
train_matrix, test_matrix, user_splits = split_data_for_evaluation(
    user_item_matrix_norm,
    test_ratio=0.2
)

  Train matrix shape: (6383, 6036)
  Test matrix shape: (6383, 6036)


In [21]:
# Train model on training set
item_similarity = cosine_similarity(train_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)
print(f"Item similarity matrix created: {item_similarity_df.shape}")

Item similarity matrix created: (6036, 6036)


In [ ]:
# Recommendation function
def get_recommendations(user_id, num_recommendations=10):

    # Check if user exists
    if user_id not in train_matrix.index:
        return []

    # Get user's interactions from training data
    user_interactions = train_matrix.loc[user_id]

    # Calculate scores for all items
    item_scores = item_similarity_df.dot(user_interactions)

    # Don't recommend items user already interacted with
    item_scores[user_interactions > 0] = 0

    # Get items with positive scores
    potential_recommendations = item_scores[item_scores > 0]

    # Sort and get top N
    top_items = potential_recommendations.sort_values(ascending=False).head(num_recommendations)

    return top_items.index.tolist()


In [25]:
# Evaluate Model
# Test with different values of K
k_values = [1, 5, 10]

# Store results for each K
results = {
    'precision': {k: [] for k in k_values},
    'recall': {k: [] for k in k_values},
    'f1': {k: [] for k in k_values}
}

print("EVALUATING MODEL")

# Only evaluate users who have test items
users_with_test_items = [
    user_id for user_id, splits in user_splits.items()
    if len(splits['test']) > 0
]

print(f"Evaluating {len(users_with_test_items)} users...\n")

# Evaluate each user
for user_id in users_with_test_items:
    # Get what items the user actually liked (test set)
    actual_liked_items = user_splits[user_id]['test']

    # Get recommendations for this user
    recommended_items = get_recommendations(user_id, num_recommendations=max(k_values))

    # If no recommendations, all metrics are 0
    if len(recommended_items) == 0:
        for k in k_values:
            results['precision'][k].append(0.0)
            results['recall'][k].append(0.0)
            results['f1'][k].append(0.0)
        continue

    # Calculate metrics for each K value
    for k in k_values:
        # Calculate precision and recall
        prec = precision_at_k(recommended_items, actual_liked_items, k)
        rec = recall_at_k(recommended_items, actual_liked_items, k)
        f1 = f1_score_at_k(prec, rec)

        # Store results
        results['precision'][k].append(prec)
        results['recall'][k].append(rec)
        results['f1'][k].append(f1)


EVALUATING MODEL
Evaluating 6383 users...



In [30]:
print("RESULTS")

for k in k_values:
    avg_precision = np.mean(results['precision'][k])
    avg_recall = np.mean(results['recall'][k])
    avg_f1 = np.mean(results['f1'][k])

    print(f"K = {k}:")
    print(f"  Average Precision: {avg_precision:.4f}")
    print(f"  Average Recall:    {avg_recall:.4f}")
    print(f"  Average F1-Score:  {avg_f1:.4f}")


RESULTS
K = 1:
  Average Precision: 0.0157
  Average Recall:    0.0153
  Average F1-Score:  0.0154
K = 5:
  Average Precision: 0.0078
  Average Recall:    0.0369
  Average F1-Score:  0.0127
K = 10:
  Average Precision: 0.0051
  Average Recall:    0.0476
  Average F1-Score:  0.0091
